In [1]:
# Cell 0 — Setup
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyod'])

import os, glob, json, warnings, urllib.request
import numpy as np
import pandas as pd
import joblib

from pyod.models.ecod import ECOD
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve,
)

warnings.filterwarnings('ignore')
RANDOMSTATE = 42
np.random.seed(RANDOMSTATE)

# Drive paths
RUNID = 'ensemble_run_001'
DRIVEROOT = '/content/drive/MyDrive/tsad_ensemble_runs'
NOTEBOOKTAG = 'ecod'

RUNDIR = os.path.join(DRIVEROOT, RUNID, NOTEBOOKTAG)
ARTIFACTDIR = os.path.join(RUNDIR, 'artifacts')
PREDICTIONSDIR = os.path.join(RUNDIR, 'predictions')
CACHEDIR = os.path.join(DRIVEROOT, '_cache')

for d in [ARTIFACTDIR, PREDICTIONSDIR, CACHEDIR]:
    os.makedirs(d, exist_ok=True)

print('RUNDIR       :', RUNDIR)
print('ARTIFACTDIR  :', ARTIFACTDIR)
print('PREDICTIONSDIR:', PREDICTIONSDIR)
print('pyod ECOD loaded successfully')


Mounted at /content/drive
RUNDIR       : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/ecod
ARTIFACTDIR  : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/ecod/artifacts
PREDICTIONSDIR: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/ecod/predictions
pyod ECOD loaded successfully


In [2]:
# Cell 1 — Dataset Paths

MYDRIVE = '/content/drive/MyDrive'
CREDITCARDPATH = os.path.join(MYDRIVE, 'creditcard.csv')

NAB_CANDIDATES = [
    os.path.join(MYDRIVE, 'NAB Dataset'),
    os.path.join(MYDRIVE, 'NAB'),
    os.path.join(MYDRIVE, 'datasets', 'NAB Dataset'),
    os.path.join(MYDRIVE, 'datasets', 'NAB'),
]

def resolve_nab_root(candidates):
    for c in candidates:
        if os.path.isdir(c):
            csvs = glob.glob(os.path.join(c, '**', '*.csv'), recursive=True)
            if len(csvs) > 0:
                return c
    return None

NABROOT = resolve_nab_root(NAB_CANDIDATES)

# NAB labels
NABLABELSLOCAL = os.path.join(CACHEDIR, 'nab_combined_windows.json')
NABLABELSURL = 'https://raw.githubusercontent.com/numenta/NAB/master/labels/combined_windows.json'
if not os.path.exists(NABLABELSLOCAL):
    urllib.request.urlretrieve(NABLABELSURL, NABLABELSLOCAL)
with open(NABLABELSLOCAL) as f:
    NABWINDOWSMAP = json.load(f)

assert os.path.exists(CREDITCARDPATH), f'creditcard.csv not found at {CREDITCARDPATH}'
assert NABROOT is not None, 'NAB root not found.'

nab_csvs = [f for f in glob.glob(os.path.join(NABROOT, '**', '*.csv'), recursive=True) if 'README' not in f]
print(f'Credit Card : {CREDITCARDPATH}')
print(f'NAB root    : {NABROOT} ({len(nab_csvs)} CSVs)')
print(f'NAB labels  : {len(NABWINDOWSMAP)} entries')


Credit Card : /content/drive/MyDrive/creditcard.csv
NAB root    : /content/drive/MyDrive/NAB Dataset (58 CSVs)
NAB labels  : 58 entries


In [3]:
# Cell 2 — Shared Utilities

def keep_runs(y, min_len=3):
    y = np.asarray(y, dtype=np.int8).copy()
    n = len(y)
    i = 0
    while i < n:
        if y[i] == 1:
            j = i
            while j < n and y[j] == 1:
                j += 1
            if (j - i) < min_len:
                y[i:j] = 0
            i = j
        else:
            i += 1
    return y

def point_adjust(y_true, y_pred):
    yt = np.asarray(y_true, dtype=np.int8)
    yp = np.asarray(y_pred, dtype=np.int8).copy()
    n = len(yt)
    i = 0
    while i < n:
        if yt[i] == 1:
            j = i
            while j < n and yt[j] == 1:
                j += 1
            if yp[i:j].any():
                yp[i:j] = 1
            i = j
        else:
            i += 1
    return yp

def best_f1_threshold(y, scores, ngrid=300, qlo=0.50, qhi=0.999, min_run=0):
    y = np.asarray(y, dtype=int)
    scores = np.asarray(scores, dtype=float)
    if y.sum() == 0:
        return float(np.percentile(scores, 99.5)), 0.0
    qs = np.linspace(qlo, qhi, ngrid)
    thr_list = np.unique(np.quantile(scores, qs))
    best_t, best_f = float(thr_list[-1]), -1.0
    for t in thr_list:
        p = (scores >= t).astype(int)
        if min_run > 0:
            p = keep_runs(p, min_len=min_run)
        f = f1_score(y, p, zero_division=0)
        if f > best_f:
            best_f, best_t = float(f), float(t)
    return best_t, best_f

def compute_metrics(y, pred, scores=None, prefix=''):
    m = {
        f'{prefix}precision': float(precision_score(y, pred, zero_division=0)),
        f'{prefix}recall': float(recall_score(y, pred, zero_division=0)),
        f'{prefix}f1': float(f1_score(y, pred, zero_division=0)),
    }
    if scores is not None and len(np.unique(y)) == 2:
        m[f'{prefix}rocauc'] = float(roc_auc_score(y, scores))
        m[f'{prefix}prauc'] = float(average_precision_score(y, scores))
    else:
        m[f'{prefix}rocauc'] = float('nan')
        m[f'{prefix}prauc'] = float('nan')
    return m

print('Utilities ready.')


Utilities ready.


In [4]:
# Cell 3 — Credit Card ECOD

print('Loading Credit Card data...')
df = pd.read_csv(CREDITCARDPATH).dropna().reset_index(drop=True)
y_all = df['Class'].astype(int).values

# Feature engineering (same as IF improved for consistency)
X_df = df.copy()
X_df['hoursin'] = np.sin(2 * np.pi * X_df['Time'] / 86400.0)
X_df['hourcos'] = np.cos(2 * np.pi * X_df['Time'] / 86400.0)
X_df['Amountlog'] = np.log1p(X_df['Amount'])
X_df['AmountSq'] = X_df['Amount'] ** 2
for v in ['V1', 'V3', 'V4', 'V7', 'V10', 'V12', 'V14', 'V17']:
    X_df[f'{v}_abs'] = X_df[v].abs()

feature_cols = ([f'V{i}' for i in range(1, 29)]
                + ['Amountlog', 'AmountSq', 'hoursin', 'hourcos']
                + [f'{v}_abs' for v in ['V1', 'V3', 'V4', 'V7', 'V10', 'V12', 'V14', 'V17']])

X_all = X_df[feature_cols].values.astype(np.float64)

# Same split as all other models: 80/20 stratified, RANDOM_STATE=42
idx = np.arange(len(y_all), dtype=np.int64)
idx_tune, idx_test = train_test_split(idx, test_size=0.20, stratify=y_all, random_state=RANDOMSTATE)
idx_train, idx_val = train_test_split(idx_tune, test_size=0.25, stratify=y_all[idx_tune], random_state=RANDOMSTATE)

X_train_normal = X_all[idx_train][y_all[idx_train] == 0]
X_val = X_all[idx_val]
y_val = y_all[idx_val]
X_test = X_all[idx_test]
y_test = y_all[idx_test]

print(f'Train normal: {len(X_train_normal):,} rows')
print(f'Validation  : {len(X_val):,} rows ({y_val.sum()} frauds)')
print(f'Test        : {len(X_test):,} rows ({y_test.sum()} frauds)')

# Fit ECOD on normal training data
print('Fitting ECOD (parameter-free)...')
ecod = ECOD()
ecod.fit(X_train_normal)

# Score all sets
scores_val = ecod.decision_function(X_val)
scores_test = ecod.decision_function(X_test)

# Threshold from PR-curve on validation
prec, rec, thr = precision_recall_curve(y_val, scores_val)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
if len(thr) > 0:
    best_idx = int(np.nanargmax(f1s))
    cc_threshold = float(thr[best_idx])
    print(f'Threshold from PR-curve: {cc_threshold:.4f} (val F1={f1s[best_idx]:.4f})')
else:
    cc_threshold = float(np.percentile(scores_val, 99.5))
    print(f'Fallback threshold: {cc_threshold:.4f}')

# Test predictions
cc_preds = (scores_test >= cc_threshold).astype(np.int8)
cc_scores = scores_test.astype(np.float32)
cc_ytrue = y_test.astype(np.int8)

strict = compute_metrics(cc_ytrue, cc_preds, cc_scores)

cc_results = {
    'dataset': 'creditcard',
    'protocol': 'semi-supervised strict holdout (20% test)',
    **strict,
    'threshold': cc_threshold,
    'n_anomalies_true': int(cc_ytrue.sum()),
    'n_anomalies_pred': int(cc_preds.sum()),
}

# Save artifact
joblib.dump({
    'model': ecod,
    'feature_cols': feature_cols,
    'threshold': cc_threshold,
    'test_indices': idx_test,
}, os.path.join(ARTIFACTDIR, 'ecod_creditcard.joblib'))

print()
print('=' * 60)
print('CREDIT CARD ECOD RESULTS')
print('=' * 60)
for k, v in cc_results.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')


Loading Credit Card data...
Train normal: 170,588 rows
Validation  : 56,962 rows (99 frauds)
Test        : 56,962 rows (98 frauds)
Fitting ECOD (parameter-free)...
Threshold from PR-curve: 159.7351 (val F1=0.3704)

CREDIT CARD ECOD RESULTS
  dataset                  : creditcard
  protocol                 : semi-supervised strict holdout (20% test)
  precision                : 0.342105
  recall                   : 0.530612
  f1                       : 0.416000
  rocauc                   : 0.961591
  prauc                    : 0.353613
  threshold                : 159.735134
  n_anomalies_true         : 98
  n_anomalies_pred         : 152


In [5]:
# Cell 6 — Export for Coordinator (Credit Card only)

exported = {}
summaryrows = []

# --- Credit Card ---
cc_bundle = {
    'dataset': 'creditcard',
    'model': 'ecod',
    'protocol': 'semi-supervised strict holdout',
    'entities': {
        'creditcard': {
            'entityid': 'creditcard',
            'scoresfull': cc_scores,
            'yfull': cc_ytrue,
            'predfull': cc_preds,
            'rowid': np.arange(len(cc_ytrue), dtype=np.int64),
            'originalrowid': idx_test.astype(np.int64),
            'threshold': cc_threshold,
        }
    }
}

ccpath = os.path.join(PREDICTIONSDIR, 'ecod_creditcard_strict.joblib')
joblib.dump(cc_bundle, ccpath)
exported['creditcard'] = ccpath
summaryrows.append({'dataset': 'creditcard', **{k: v for k, v in cc_results.items() if k != 'dataset'}})
print('Saved', ccpath)

# --- Summary ---
summary = pd.DataFrame(summaryrows)
summarypath = os.path.join(PREDICTIONSDIR, 'ecod_summary.csv')
summary.to_csv(summarypath, index=False)
print('Saved', summarypath)
display(summary)

# --- Manifest ---
manifest = {
    'runid': RUNID,
    'driveroot': DRIVEROOT,
    'notebooktag': NOTEBOOKTAG,
    'modelfamily': 'ecod',
    'exportprotocol': 'ensembleexportv2',
    'artifactsdir': ARTIFACTDIR,
    'predictionsdir': PREDICTIONSDIR,
    'exports': exported,
    'summarycsv': summarypath,
    'creditcard_originalrowid_included': True,
    'datasets': ['creditcard'],
}

manifestpath = os.path.join(PREDICTIONSDIR, 'ecod_manifest.json')
with open(manifestpath, 'w') as f:
    json.dump(manifest, f, indent=2)
print('Saved', manifestpath)

print()
print('Coordinator-ready files in:', PREDICTIONSDIR)
print('Done.')

Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/ecod/predictions/ecod_creditcard_strict.joblib
Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/ecod/predictions/ecod_summary.csv


,dataset,protocol,precision,recall,f1,rocauc,prauc,threshold,n_anomalies_true,n_anomalies_pred
0,creditcard,semi-supervised strict holdout (20% test),0.342105,0.530612,0.416,0.961591,0.353613,159.735134,98,152


Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/ecod/predictions/ecod_manifest.json

Coordinator-ready files in: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/ecod/predictions
Done.
